# RutAI — Simulador UCB + Entropía

**Artículo:** Comparación de algoritmos de bandido para recomendación de rutas  
**Versión:** 2 (corrección reward_rate en misma traza)  
**Semillas:** 100 | **Horizonte:** T = 500

Este notebook es el artefacto reproducible para Zenodo. Ejecutar en orden.

## 1. Instalación de dependencias

In [ ]:
# !pip install numpy matplotlib scipy nbformat
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'numpy', 'matplotlib', 'scipy', 'nbformat', '-q'])

## 2. Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, math, random, csv
from pathlib import Path

SEED_BASE   = 42
N_SEEDS     = 100
HORIZON     = 500
TIPOS_RUTA  = ['fastest', 'shortest', 'recommended']
REWARD_PROBS = {"fastest": 0.4, "shortest": 0.5, "recommended": 0.65}
OPTIMAL_ARM  = "recommended"
OPTIMAL_PROB = REWARD_PROBS[OPTIMAL_ARM]
print("Config OK — brazos:", TIPOS_RUTA)
print("Probabilidades de recompensa:", REWARD_PROBS)

## 3. Definición del entorno y algoritmos
Se copia la lógica de `app/services/ucb_service.py`.

In [ ]:
class BanditEnv:
    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)
    def pull(self, arm):
        return int(self.rng.random() < REWARD_PROBS[arm])

class UCBRutAI:
    def __init__(self, seed, lambda_ent=0.1):
        self.rng = random.Random(seed)
        self.lambda_ent = lambda_ent
        self.total_usos    = {t: 0 for t in TIPOS_RUTA}
        self.total_rewards = {t: 0 for t in TIPOS_RUTA}
    def select(self):
        total = sum(self.total_usos.values())
        if total < len(TIPOS_RUTA):
            pend = [t for t in TIPOS_RUTA if self.total_usos[t] == 0]
            if pend: return self.rng.choice(pend)
        for t in TIPOS_RUTA:
            if self.total_usos[t] == 0: return t
        best, bs = None, -float('inf')
        for t in TIPOS_RUTA:
            avg  = self.total_rewards[t] / self.total_usos[t]
            conf = math.sqrt(2 * math.log(total) / self.total_usos[t])
            p    = avg
            H    = 0.0 if p<=0 or p>=1 else -p*math.log2(p)-(1-p)*math.log2(1-p)
            s    = avg + conf + self.lambda_ent * H
            if s > bs: bs, best = s, t
        return best or 'fastest'
    def update(self, arm, reward):
        self.total_usos[arm]    += 1
        self.total_rewards[arm] += reward
print("Clases definidas OK")

## 4. Simulación
**CORRECCIÓN v2:** `reward_rate` se mide en las últimas 100 iteraciones de la **misma** traza, eliminando el sesgo de re-simulación.

In [ ]:
def simulate(policy_factory, n_seeds=N_SEEDS, horizon=HORIZON):
    all_regrets, all_rates = [], []
    for s in range(n_seeds):
        env    = BanditEnv(SEED_BASE + s)
        policy = policy_factory(SEED_BASE + s)
        cum, run, last100 = 0.0, [], []
        for t in range(horizon):
            arm = policy.select()
            r   = env.pull(arm)
            cum += OPTIMAL_PROB - REWARD_PROBS[arm]
            run.append(cum)
            if t >= horizon - 100: last100.append(r)
            policy.update(arm, r)
        all_regrets.append(run)
        all_rates.append(float(np.mean(last100)))
    arr   = np.array(all_regrets)
    rng_b = np.random.default_rng(0)
    bidx  = rng_b.choice(n_seeds, size=(1000, n_seeds), replace=True)
    bmean = np.array([arr[bidx[i]].mean(axis=0) for i in range(1000)])
    ci_l  = np.percentile(bmean, 2.5,  axis=0)
    ci_u  = np.percentile(bmean, 97.5, axis=0)
    return {
        'regret_mean': arr.mean(axis=0), 'ci_lower': ci_l, 'ci_upper': ci_u,
        'final_regret': float(arr[:,-1].mean()), 'final_ci': float((ci_u-ci_l)[-1]/2),
        'reward_rate': float(np.mean(all_rates)), 'reward_std': float(np.std(all_rates)),
    }
print("Función simulate() OK")

## 5. Resultados — Tabla 4

In [ ]:
import importlib, sys

# Definir baselines inline para el notebook
class RandomPolicy:
    def __init__(self, s): self.rng = random.Random(s)
    def select(self): return self.rng.choice(TIPOS_RUTA)
    def update(self, a, r): pass

class GreedyPolicy:
    def __init__(self, s):
        self.rng = random.Random(s)
        self.u = {t:0 for t in TIPOS_RUTA}
        self.rw = {t:0 for t in TIPOS_RUTA}
    def select(self):
        for t in TIPOS_RUTA:
            if self.u[t]==0: return t
        return max(TIPOS_RUTA, key=lambda t: self.rw[t]/self.u[t])
    def update(self, a, r): self.u[a]+=1; self.rw[a]+=r

class EG:
    def __init__(self, eps, s):
        self.eps=eps; self.rng=random.Random(s)
        self.u={t:0 for t in TIPOS_RUTA}; self.rw={t:0 for t in TIPOS_RUTA}
    def select(self):
        for t in TIPOS_RUTA:
            if self.u[t]==0: return t
        return self.rng.choice(TIPOS_RUTA) if self.rng.random()<self.eps else max(TIPOS_RUTA,key=lambda t:self.rw[t]/self.u[t])
    def update(self, a, r): self.u[a]+=1; self.rw[a]+=r

class TS:
    def __init__(self, s):
        self.rng=np.random.default_rng(s)
        self.alpha={t:1 for t in TIPOS_RUTA}; self.beta={t:1 for t in TIPOS_RUTA}
    def select(self):
        return max({t:self.rng.beta(self.alpha[t],self.beta[t]) for t in TIPOS_RUTA}, key=lambda t:0)
    def update(self, a, r):
        if r: self.alpha[a]+=1
        else: self.beta[a]+=1

policies_nb = {
    'Random':           lambda s: RandomPolicy(s),
    'Greedy':           lambda s: GreedyPolicy(s),
    'ε-greedy (0.1)':  lambda s: EG(0.1, s),
    'ε-greedy (0.2)':  lambda s: EG(0.2, s),
    'Thompson':        lambda s: TS(s),
    'UCB+Entropía':    lambda s: UCBRutAI(s),
}

results_nb = {}
for name, fac in policies_nb.items():
    print(f'Simulando {name}...', end=' ')
    results_nb[name] = simulate(fac)
    r = results_nb[name]
    print(f"regret={r['final_regret']:.2f}±{r['final_ci']:.2f}  rate={r['reward_rate']:.3f}")

## 6. Visualización

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
t_arr = np.arange(1, HORIZON+1)
for name, r in results_nb.items():
    ax.plot(t_arr, r['regret_mean'], label=name, lw=2.5 if 'UCB' in name else 1.5)
    ax.fill_between(t_arr, r['ci_lower'], r['ci_upper'], alpha=0.1)
ax.set_xlabel('Interacciones (t)'); ax.set_ylabel('Regret acumulado')
ax.set_title(f'Regret acumulado — T={HORIZON}, {N_SEEDS} semillas, IC95% bootstrap')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Cómo citar

```
Autor, A. (2025). RutAI UCB Simulation [Software]. Zenodo. https://doi.org/XXXX
```

Sustituir `XXXX` por el DOI asignado en Zenodo tras la subida.